<a href="https://colab.research.google.com/github/ypg1um-arch/SAU_ML_TASKS/blob/main/PMMHA_Ablation_Study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Clone the repository
!git clone https://github.com/SinaTabakhi/MAGNET.git
%cd MAGNET

# 2. Detect Colab's current PyTorch version dynamically
import torch
torch_version = torch.__version__.split('+')[0]
cuda_version = torch.version.cuda.replace('.', '')
print(f"Colab is using Torch {torch_version} with CUDA {cuda_version}")

# 3. Install PyG binaries that match Colab's environment exactly
!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-{torch_version}+cu{cuda_version}.html
!pip install torch-geometric==2.4.0

# 4. Install the remaining requirements
!pip install lightning==2.1.3 pandas matplotlib umap-learn yacs comet_ml "ray[tune]"

Cloning into 'MAGNET'...
remote: Enumerating objects: 202, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 202 (delta 8), reused 0 (delta 0), pack-reused 167 (from 1)
Receiving objects: 100% (202/202), 53.31 MiB | 22.45 MiB/s, done.
Resolving deltas: 100% (49/49), done.
Updating files: 100% (151/151), done.
/content/MAGNET
Colab is using Torch 2.11.0 with CUDA 128
Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 98.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 84.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 76.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

In [3]:
%cd /content/MAGNET

/content/MAGNET


In [ ]:
!python main_inference.py --cfg configs/MAGNET_BRCA.yaml

In [ ]:
#base_models.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import Linear
from torch_geometric.nn.inits import glorot
from torch_geometric.nn.conv import MessagePassing
from torch import Tensor


class MLPEncoder(nn.Module):
    def __init__(self, in_dims, hid_dims, dropout_rate: float = 0.0, negative_slope: float = 0.2):
        super().__init__()
        self.encoder_layers = nn.Sequential(
            Linear(in_dims, hid_dims, weight_initializer='glorot', bias_initializer='zeros'),
            nn.LeakyReLU(negative_slope),
            nn.Dropout(p=dropout_rate),
            Linear(hid_dims, hid_dims, weight_initializer='glorot', bias_initializer='zeros')
        )

    def forward(self, x):
        return self.encoder_layers(x)


# -------------------------------------------------------------------------------
# Generalized fuse() function
# -------------------------------------------------------------------------------
def fuse(x_proj, mask, mode="original", att_lin=None, shared_weights=None, external_confidence=None):
    """
    Standalone fusion function for MAGNET Ablation Study.
    """
    batch_size, num_heads, num_modalities, head_dims = x_proj.size()

    if mode == "original":
        assert att_lin is not None, "att_lin parameter is required for 'original' mode"
        att_scores = torch.matmul(x_proj, att_lin.transpose(-1, -2)).squeeze(-1)
        att_scores = att_scores.masked_fill(mask.unsqueeze(1) == 0, float('-inf'))
        att_weights = torch.softmax(att_scores, dim=-1)

    elif mode == "equal":
        counts = mask.sum(dim=1, keepdim=True).clamp(min=1)
        equal_weights = mask / counts
        att_weights = equal_weights.unsqueeze(1).expand(-1, num_heads, -1)

    elif mode == "shared":
        assert shared_weights is not None, "shared_weights parameter is required for 'shared' mode"
        shared_scores = shared_weights.view(1, 1, num_modalities).expand(batch_size, num_heads, -1)
        shared_scores = shared_scores.masked_fill(mask.unsqueeze(1) == 0, float('-inf'))
        att_weights = torch.softmax(shared_scores, dim=-1)

    elif mode == "external":
        assert external_confidence is not None, "external_confidence is required for 'external' mode"
        # external_confidence shape expected: [batch_size, num_modalities]
        ext_conf = external_confidence.unsqueeze(1).expand(-1, num_heads, -1)
        ext_conf = ext_conf * mask.unsqueeze(1) # Respect patient mask
        # Normalize weights so they sum to 1.0 per head, acting like a standard attention distribution
        att_weights = ext_conf / ext_conf.sum(dim=-1, keepdim=True).clamp(min=1e-9)

    else:
        raise ValueError(f"Unknown mode: {mode}. Choose 'equal', 'original', 'shared', or 'external'.")

    att_weights = att_weights * mask.unsqueeze(1)
    fused_embeddings = torch.sum(att_weights.unsqueeze(-1) * x_proj, dim=2)

    return fused_embeddings, att_weights


class MultiHeadAttentionLayer(nn.Module):
    def __init__(self, hid_dims, num_heads, num_modalities=3):
        super().__init__()
        self.num_heads = num_heads
        self.head_dims = hid_dims // num_heads
        assert self.head_dims * num_heads == hid_dims, "hid_dims must be divisible by num_heads"

        self.lin_proj = Linear(hid_dims, hid_dims, bias=False, weight_initializer='glorot')
        self.att_lin = nn.Parameter(torch.empty(num_heads, 1, self.head_dims))
        self.out_proj = Linear(hid_dims, hid_dims, bias=False, weight_initializer='glorot')

        # Shared learnable weights
        self.shared_weights = nn.Parameter(torch.zeros(num_modalities))

        self.reset_parameters()

    def reset_parameters(self):
        self.lin_proj.reset_parameters()
        self.out_proj.reset_parameters()
        glorot(self.att_lin)
        nn.init.normal_(self.shared_weights, mean=0.0, std=0.1)

    def forward(self, x, mask, mode="original", external_confidence=None):
        batch_size, num_modalities, hid_dims = x.size()

        # 1. Linear projection
        x_proj = self.lin_proj(x).view(batch_size, num_modalities, self.num_heads, self.head_dims)
        x_proj = x_proj.permute(0, 2, 1, 3)

        # 2. Call standalone fuse function dynamically based on mode argument
        fused_embeddings, att_weights = fuse(
            x_proj=x_proj,
            mask=mask,
            mode=mode,
            att_lin=self.att_lin,
            shared_weights=self.shared_weights,
            external_confidence=external_confidence
        )

        # 3. Concatenate heads and project output
        fused_embeddings = fused_embeddings.view(batch_size, -1)
        output = self.out_proj(fused_embeddings)

        return output, att_weights


# EdgeSAGEConv and GNNDecoder remain unchanged
class EdgeSAGEConv(MessagePassing):
    def __init__(self, in_channels: int, out_channels: int, aggr="mean", bias: bool=True, edge_dim: int=None, **kwargs):
        super().__init__(aggr, **kwargs)
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.edge_dim = edge_dim
        in_channels = (in_channels, in_channels)

        if self.edge_dim is not None:
            self.lin_msg = Linear(in_channels[0] + self.edge_dim, in_channels[0], weight_initializer='glorot', bias_initializer='zeros', bias=True)

        self.lin = Linear(in_channels[0], in_channels[0], weight_initializer='glorot', bias_initializer='zeros', bias=True)
        self.lin_l = Linear(in_channels[0], out_channels, weight_initializer='glorot', bias_initializer='zeros', bias=bias)
        self.lin_r = Linear(in_channels[1], out_channels, weight_initializer='glorot', bias_initializer='zeros', bias=False)
        self.act_msg = nn.ReLU()
        self.reset_parameters()

    def reset_parameters(self):
        super().reset_parameters()
        self.lin.reset_parameters()
        self.lin_l.reset_parameters()
        self.lin_r.reset_parameters()
        if self.edge_dim is not None:
            self.lin_msg.reset_parameters()

    def forward(self, x, edge_index, edge_attr=None):
        if isinstance(x, Tensor):
            x = (x, x)
        x = (self.lin(x[0]).relu(), x[1])
        out = self.propagate(edge_index, x=x, edge_attr=edge_attr)
        out = self.lin_l(out)
        out = out + self.lin_r(x[1])
        return out

    def message(self, x_j, edge_attr=None):
        if edge_attr is not None and self.edge_dim is not None:
            if edge_attr.dim() == 1:
                edge_attr = edge_attr.unsqueeze(-1)
            msg = torch.cat([x_j, edge_attr], dim=-1)
            return self.act_msg(self.lin_msg(msg))
        return x_j


class GNNDecoder(nn.Module):
    def __init__(self, hid_dims, out_dims, num_layers, dropout_rate: float = 0.0, negative_slope: float = 0.2):
        super().__init__()
        self.dropout_rate = dropout_rate
        self.negative_slope = negative_slope
        self.conv_layers = nn.ModuleList([EdgeSAGEConv(hid_dims, hid_dims, edge_dim=1) for _ in range(num_layers)])
        self.decoder_layers = nn.Sequential(
            Linear(hid_dims, hid_dims, weight_initializer='glorot', bias_initializer='zeros'),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate)
        )
        self.final_layer = Linear(hid_dims, out_dims, weight_initializer='glorot', bias_initializer='zeros')

    def forward(self, x, edge_index, edge_attr=None, return_embedding=False):
        for conv in self.conv_layers:
            x = F.dropout(F.relu(conv(x, edge_index, edge_attr=edge_attr)), p=self.dropout_rate, training=self.training)

        x = self.decoder_layers(x)
        logits = self.final_layer(x)
        if return_embedding:
            return logits, x
        return logits

In [ ]:
#trainer.py
from typing import List
import torch
import torch.nn as nn
from torch.nn import CrossEntropyLoss
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR
import lightning as L

from model.base_models import MLPEncoder, MultiHeadAttentionLayer, GNNDecoder
from dataloader.dataloader import MultiomicsDataset
from utils import align_modalities, evaluate_classification_performance


class MAGNETTrainer(L.LightningModule):
    def __init__(
        self,
        dataset: MultiomicsDataset,
        unimodal_encoders: List[MLPEncoder],
        attention_layer: MultiHeadAttentionLayer,
        gnn_decoder: GNNDecoder,
        loss_fn: CrossEntropyLoss,
        lr: float,
        wd: float,
        fusion_mode: str = "original",
        target_modality: int = None,
        gamma: float = 1.0
    ):
        super().__init__()
        self.save_hyperparameters("lr", "wd", "fusion_mode", "target_modality", "gamma")
        self.dataset = dataset
        self.unimodal_encoders = nn.ModuleList(unimodal_encoders)
        self.attention_layer = attention_layer
        self.gnn_decoder = gnn_decoder
        self.loss_fn = loss_fn
        self.lr = lr
        self.wd = wd

        # Ablation specific setup
        self.fusion_mode = fusion_mode
        self.target_modality = target_modality
        self.gamma = gamma

        self.train_embeddings_to_plot = {}
        self.train_labels_to_plot = {}
        self.test_embeddings_to_plot = {}
        self.test_labels_to_plot = {}

    def configure_optimizers(self):
        optimizer = Adam(self.parameters(), lr=self.lr, weight_decay=self.wd)
        scheduler = StepLR(optimizer, step_size=20, gamma=0.8)
        return {'optimizer': optimizer, 'lr_scheduler': {'scheduler': scheduler, 'interval': 'epoch', 'frequency': 1}}

    def similarity_kl_loss(self, similarities, embeddings, mask=None, alpha=1.0, eps=1e-12):
        device = embeddings.device
        n = embeddings.size(0)
        sum_sq = torch.sum(embeddings ** 2, dim=1)
        dist_sq = sum_sq.unsqueeze(1) + sum_sq.unsqueeze(0) - 2 * torch.matmul(embeddings, embeddings.T)
        q_matrix = torch.pow(1 + dist_sq / alpha, -(alpha + 1) / 2) * (1 - torch.eye(n, device=device))

        if mask is not None:
            q_matrix = q_matrix.masked_fill(mask == 0, 0.0)
            similarities = similarities.masked_fill(mask == 0, 0.0)

        q_distribution = torch.clamp(q_matrix / (q_matrix.sum() + eps), min=eps)
        p_distribution = torch.clamp(similarities / (similarities.sum() + eps), min=eps)
        return torch.sum(p_distribution * torch.log(p_distribution / q_distribution))

    def forward(self, x, mode, return_embedding=False):
        data, data_indices, modality_mask, labels = x
        encoded_modalities = [encoder(data[i]) for i, encoder in enumerate(self.unimodal_encoders)]

        aligned_modalities = align_modalities(encoded_modalities, data_indices, labels)
        modality_embeddings = torch.stack(aligned_modalities, dim=1)

        # Build external confidence tensor for ablation study
        external_confidence = None
        if self.fusion_mode == "external":
            batch_size = modality_embeddings.size(0)
            # Default weights to 1.0 (equal importance before downweighting)
            external_confidence = torch.ones((batch_size, self.dataset.num_omics), device=self.device)
            if self.target_modality is not None:
                external_confidence[:, self.target_modality] = self.gamma

        fused_embeddings, _ = self.attention_layer(
            modality_embeddings,
            modality_mask,
            mode=self.fusion_mode,
            external_confidence=external_confidence
        )

        graph_data, similarities, edge_mask = self.dataset.build_graph_data(fused_embeddings, mode)

        if return_embedding:
            output, gnn_embeddings = self.gnn_decoder(graph_data.x, graph_data.edge_index, graph_data.edge_attr, return_embedding=True)
            return aligned_modalities, fused_embeddings, gnn_embeddings, graph_data

        output = self.gnn_decoder(graph_data.x, graph_data.edge_index, graph_data.edge_attr)
        return output, graph_data, similarities, fused_embeddings, edge_mask

    def training_step(self, batch, batch_idx):
        output, graph_data, similarities, fused_embeddings, edge_mask = self(batch, mode="train")
        train_kl_loss = self.similarity_kl_loss(similarities, fused_embeddings, edge_mask)

        mask = graph_data.train_mask
        y_true, y_logit = graph_data.y.squeeze()[mask], output[mask]
        train_cls_loss = self.loss_fn(y_logit, y_true)
        total_loss = train_cls_loss + 0.1 * train_kl_loss

        self.log('train_kl_loss', train_kl_loss.detach().cpu(), prog_bar=False)
        self.log('train_cls_loss', train_cls_loss.detach().cpu(), prog_bar=False)
        self.log('train_total_loss', total_loss.detach().cpu(), prog_bar=False)
        return total_loss

    def test_step(self, batch, batch_idx):
        output, graph_data, similarities, fused_embeddings, edge_mask = self(batch, mode="test")
        mask = graph_data.test_mask
        y_true, y_logit = graph_data.y.squeeze()[mask], output[mask]

        probs = torch.softmax(y_logit, dim=-1)
        metrics = evaluate_classification_performance(y_true, probs, self.dataset.num_classes)

        for name, value in metrics.items():
            self.log(f'test_{name}', value, prog_bar=False)

    # Simplified evaluation methods and data loaders (kept minimal for inference optimization)
    def _custom_data_loader(self): return self.dataset
    def train_dataloader(self): return self._custom_data_loader()
    def test_dataloader(self): return self._custom_data_loader()

In [ ]:
#main_inference.py
import os
import gc
import warnings
import argparse
import pandas as pd
import numpy as np
import lightning as L
from lightning.pytorch.callbacks import Callback
from torch.nn import CrossEntropyLoss
import torch

from utils import seed_everything
from dataloader.dataloader import MultiomicsDataset
from model.base_models import MLPEncoder, GNNDecoder, MultiHeadAttentionLayer
from trainer.trainer import MAGNETTrainer
from configs.config import get_cfg_defaults

class PrintEpochCallback(Callback):
    """Prints training progress strictly 1/10th of the max epochs to reduce clutter."""
    def on_train_epoch_end(self, trainer, pl_module):
        interval = max(1, trainer.max_epochs // 10)
        if (trainer.current_epoch + 1) % interval == 0:
            print(f"      [Progress] Epoch {trainer.current_epoch + 1}/{trainer.max_epochs} completed.")

def arg_parse():
    parser = argparse.ArgumentParser(description="MAGNET Full Ablation & Sensitivity Sweep")
    parser.add_argument("--cfg", required=True, help="path to config file", type=str)
    return parser.parse_args()

def run_experiment(cfg, split_folder, seed, fusion_mode, lightning_dir, target_mod_idx=None, gamma=1.0):
    """Helper function to cleanly execute a single training/testing run and free memory."""
    seed_everything(seed)

    # 1. Load Data
    multiomics = MultiomicsDataset(
        split_folder=split_folder, dataset_name=cfg.DATASET.NAME,
        modalities=cfg.DATASET.OMICS, seed=seed,
        num_classes=cfg.DATASET.NUM_CLASSES, class_names=cfg.DATASET.CLASS_NAMES,
        sparsity_rate=cfg.DATASET.SPARSITY_RATES, tune_hyperparameters=False
    )

    modality_features = [multiomics.get_data(omics_idx=i).shape[1] for i in range(len(cfg.DATASET.OMICS))]

    # 2. Build Models
    encoders = [
        MLPEncoder(in_dims=modality_features[i], hid_dims=cfg.ENCODER.HID_DIMS, dropout_rate=cfg.ENCODER.DROPOUT_RATE)
        for i in range(len(cfg.DATASET.OMICS))
    ]
    attention_layer = MultiHeadAttentionLayer(hid_dims=cfg.ENCODER.HID_DIMS, num_heads=cfg.ATTENTION.NUM_HEADS)
    decoder = GNNDecoder(hid_dims=cfg.DECODER.HID_DIMS, out_dims=cfg.DATASET.NUM_CLASSES, num_layers=cfg.DECODER.NUM_LAYERS, dropout_rate=cfg.DECODER.DROPOUT_RATE, negative_slope=cfg.DECODER.NEGATIVE_SLOPE)

    # 3. Setup Trainer
    model = MAGNETTrainer(
        dataset=multiomics, unimodal_encoders=encoders,
        attention_layer=attention_layer, gnn_decoder=decoder,
        loss_fn=CrossEntropyLoss(), lr=cfg.SOLVER.LR, wd=cfg.SOLVER.WD,
        fusion_mode=fusion_mode, target_modality=target_mod_idx, gamma=gamma
    )

    trainer = L.Trainer(
        max_epochs=cfg.SOLVER.MAX_EPOCHS,
        accelerator="auto",
        devices="auto",
        logger=False,
        enable_progress_bar=False,
        enable_model_summary=False,
        callbacks=[PrintEpochCallback()],
        default_root_dir=lightning_dir
    )

    # 4. Train & Test
    trainer.fit(model)
    trainer.test(model, verbose=False)

    # 5. Extract Metrics
    metrics = trainer.logged_metrics

    # Dynamically find the keys for accuracy and F1 to avoid case-sensitivity typos
    acc_key = next((k for k in metrics.keys() if 'acc' in k.lower()), None)
    f1_key = next((k for k in metrics.keys() if 'f1' in k.lower()), None)

    acc = metrics[acc_key].item() if acc_key else 0.0
    f1 = metrics[f1_key].item() if f1_key else 0.0

    # Print a warning if it still fails so we can see what keys actually exist
    if acc == 0.0 and f1 == 0.0:
        print(f"\n[WARNING] Could not find Acc/F1. Available metrics: {metrics.keys()}")

    # 6. Garbage Collection (Critical for preventing OOM in loops)
    del model, trainer, multiomics, encoders, attention_layer, decoder
    gc.collect()
    torch.cuda.empty_cache()

    return acc, f1

def main():
    warnings.filterwarnings(action="ignore")
    args = arg_parse()
    cfg = get_cfg_defaults()
    cfg.merge_from_file(args.cfg)
    cfg.freeze()

    lightning_dir = os.path.join(cfg.RESULT.OUTPUT_DIR, cfg.RESULT.LIGHTNING_LOG_DIR)
    if not os.path.exists(lightning_dir):
        os.makedirs(lightning_dir)
    split_folder = os.path.join(cfg.DATASET.ROOT, cfg.DATASET.SPLITS)

    seeds = [10, 20, 30, 40, 50] # 5 seeds for error bars

    print(f"\n{'='*60}")
    print(f"=== Starting Experiments on {cfg.DATASET.NAME} ===")
    print(f"{'='*60}\n")

    # ==========================================
    # DELIVERABLE 1: 3x3 Fusion Mode Ablation
    # ==========================================
    print(">>> PHASE 1: 3x3 Fusion Mode Ablation (equal, original, shared)")
    ablation_modes = ["equal", "original", "shared"]
    ablation_results = []

    for mode in ablation_modes:
        print(f"\n  [Evaluating Mode: {mode.upper()}]")
        acc_list, f1_list = [], []
        for seed in seeds:
            print(f"    -> Running Seed {seed}...")
            acc, f1 = run_experiment(cfg, split_folder, seed, mode, lightning_dir)
            acc_list.append(acc)
            f1_list.append(f1)

        mean_acc, std_acc = np.mean(acc_list), np.std(acc_list)
        mean_f1, std_f1 = np.mean(f1_list), np.std(f1_list)

        ablation_results.append({
            "Fusion Mode": mode.capitalize(),
            "Accuracy": f"{mean_acc:.4f} ± {std_acc:.4f}",
            "Macro_F1": f"{mean_f1:.4f} ± {std_f1:.4f}"
        })
        print(f"  Result -> Acc: {mean_acc:.4f}, F1: {mean_f1:.4f}")

    # ==========================================
    # DELIVERABLE 2: γ-Sensitivity Sweep
    # ==========================================
    print("\n>>> PHASE 2: γ-Sensitivity Sweep (external confidence manipulation)")
    gammas = [1.0, 0.5, 0.25, 0.0]
    target_modalities = {"DNA": 0, "mRNA": 1} # Adjust indices if dataset differs
    gamma_results = []

    for mod_name, mod_idx in target_modalities.items():
        for gamma in gammas:
            print(f"\n  [Down-weighting {mod_name} | γ = {gamma}]")
            acc_list, f1_list = [], []
            for seed in seeds:
                print(f"    -> Running Seed {seed}...")
                acc, f1 = run_experiment(cfg, split_folder, seed, "external", lightning_dir, target_mod_idx=mod_idx, gamma=gamma)
                acc_list.append(acc)
                f1_list.append(f1)

            mean_acc, std_acc = np.mean(acc_list), np.std(acc_list)
            mean_f1, std_f1 = np.mean(f1_list), np.std(f1_list)

            gamma_results.append({
                "Target Modality": mod_name,
                "Gamma": gamma,
                "Accuracy": f"{mean_acc:.4f} ± {std_acc:.4f}",
                "Macro_F1": f"{mean_f1:.4f} ± {std_f1:.4f}"
            })
            print(f"  Result -> Acc: {mean_acc:.4f}, F1: {mean_f1:.4f}")


    # ==========================================
    # FINAL TERMINAL OUTPUT & SAVING
    # ==========================================
    print(f"\n\n{'='*60}")
    print("=== FINAL DELIVERABLES ===")
    print(f"{'='*60}\n")

    # DataFrames
    df_ablation = pd.DataFrame(ablation_results)
    df_gamma = pd.DataFrame(gamma_results)

    # Print Ablation Table
    print("DELIVERABLE 1: 3x3 Seeded Ablation Table (Modes x Seeds)")
    print("-" * 60)
    print(df_ablation.to_string(index=False))
    print("\n")

    # Print Gamma Sensitivity Table
    print("DELIVERABLE 2: γ-Sensitivity Sweep Table")
    print("-" * 60)
    print(df_gamma.to_string(index=False))
    print("\n")

    # Save to disk
    ablation_csv = os.path.join(cfg.RESULT.OUTPUT_DIR, f"{cfg.DATASET.NAME}_3x3_ablation.csv")
    gamma_csv = os.path.join(cfg.RESULT.OUTPUT_DIR, f"{cfg.DATASET.NAME}_gamma_sensitivity.csv")

    df_ablation.to_csv(ablation_csv, index=False)
    df_gamma.to_csv(gamma_csv, index=False)

    print(f"Results successfully saved to:\n - {ablation_csv}\n - {gamma_csv}")

if __name__ == '__main__':
    main()